# ISS Group 24 — Two-Model Few-Shot System

**Models**:
1. **Localizer** — multi-shot OWLv2 + learned support fusion. Predicts a bounding
   box conditioned on knowing the object is present (positive-only training).
   Headline metric: **mAP@50** + **containment** (how much of the GT bbox
   the prediction encloses).
2. **Siamese** — multi-shot DINOv2-small + cross-attention head. Predicts
   `existence_prob ∈ [0, 1]` (object present in scene yes/no). Trained with
   focal-BCE + variance + decorrelation regularisers; positive:negative = 1:3.
   Headline metric: **AUROC**, with FPR closely watched (lower is better).

The two models are **fully independent** — train, evaluate, save, and run
inference on either alone. Combined cascaded inference (siamese → localizer)
is provided in `inference_combined.py` with a configurable threshold.

**Stages** (every (epoch, fold) checkpoint is resumable):
- Localizer: Phase 0 (zero-shot OWLv2) → L1 (fusion only) → L2 (+heads) → L3 (+LoRA)
- Siamese:   Phase 0 (zero-shot DINOv2 cosine) → S1 (head only) → S2 (+LoRA)

Set `RUN_SMOKE_TEST = True` to run a ~60s end-to-end smoke check before any
training. Adjust `SHARED_KWARGS` to override any hyperparameter.


In [1]:
import os
import sys
import shutil
import subprocess
import time
from pathlib import Path

In [2]:
SHARED_FOLDER_LINK = "https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing"
REPO_URL          = "https://github.com/pewpewnor/iss_group_24.git"
REPO_LOCAL_PATH   = "/content/iss_group_24_src"
USE_GOOGLE_COLAB  = False

# When USE_GOOGLE_COLAB is True we copy the staged dataset from Drive into
# the Colab runtime's local SSD for fast random reads. Checkpoints + analysis
# JSONs always go to the Drive folder so they survive runtime resets.
LOCAL_DATA_ROOT = "/content/iss_group_24_data"


In [3]:
if USE_GOOGLE_COLAB:
    from google.colab import drive

    def mount_drive() -> str:
        drive.mount("/content/drive")
        for candidate in [
            "/content/drive/Shareddrives/iss_group_24",
            "/content/drive/MyDrive/iss_group_24",
        ]:
            if Path(candidate).exists():
                print(f"Drive mounted. Project root: {candidate}")
                return candidate
        raise RuntimeError("iss_group_24 not found on the mounted Drive")

    DRIVE_ROOT = mount_drive()
else:
    DRIVE_ROOT = str(Path(".").resolve())
    print(f"Local mode. Project root: {DRIVE_ROOT}")

Local mode. Project root: /mnt/Onetera/iss_group_24


In [4]:
def setup_repo() -> None:
    if Path(REPO_LOCAL_PATH).exists():
        subprocess.run(["git", "-C", REPO_LOCAL_PATH, "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", REPO_LOCAL_PATH, "reset", "--hard", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth=1", REPO_URL, REPO_LOCAL_PATH], check=True)
    if REPO_LOCAL_PATH not in sys.path:
        sys.path.insert(0, REPO_LOCAL_PATH)


def install_deps() -> None:
    subprocess.run(["pip", "install", "-q",
                    "transformers>=4.40", "accelerate>=0.30", "peft>=0.10",
                    "huggingface_hub>=0.23", "matplotlib>=3.7"], check=True)


if USE_GOOGLE_COLAB:
    setup_repo()
    install_deps()

In [5]:
# ── Path layout ──────────────────────────────────────────────────────────
# DRIVE_ROOT  : project root on Google Drive (or local repo when not on Colab).
# DATA_ROOT   : where the staged dataset lives at READ time. On Colab this
#               points at the local SSD copy after the next cell mirrors it
#               from Drive (fast random reads). Off Colab it's the Drive copy.
# OUT_ROOT    : where checkpoints are written (always on Drive — survives
#               Colab runtime resets).
# ANALYSIS_ROOT : where per-(epoch, fold) JSON + plots are written (always on Drive).
DRIVE_DATA_ROOT     = f"{DRIVE_ROOT}/dataset/aggregated"
DRIVE_MANIFEST_PATH = f"{DRIVE_DATA_ROOT}/manifest.json"

OUT_ROOT      = f"{DRIVE_ROOT}/checkpoints"
ANALYSIS_ROOT = f"{DRIVE_ROOT}/analysis"

# These two get re-pointed in the next cell when USE_GOOGLE_COLAB is True.
DATA_ROOT = DRIVE_DATA_ROOT
MANIFEST  = DRIVE_MANIFEST_PATH

os.chdir(DRIVE_ROOT)
print(f"Working directory: {os.getcwd()}")
print(f"DRIVE_ROOT     = {DRIVE_ROOT}")
print(f"DATA_ROOT      = {DATA_ROOT}")
print(f"OUT_ROOT       = {OUT_ROOT}")
print(f"ANALYSIS_ROOT  = {ANALYSIS_ROOT}")

# Hard guard: when on Colab, refuse to start training unless OUT_ROOT and
# ANALYSIS_ROOT are on Drive (so every per-(epoch, fold) checkpoint survives a
# runtime reset).  This catches misconfigured paths early.
from shared.checkpoint import assert_checkpoint_root_on_drive
assert_checkpoint_root_on_drive(OUT_ROOT,      on_colab=USE_GOOGLE_COLAB)
assert_checkpoint_root_on_drive(ANALYSIS_ROOT, on_colab=USE_GOOGLE_COLAB)


Working directory: /mnt/Onetera/iss_group_24
DRIVE_ROOT     = /mnt/Onetera/iss_group_24
DATA_ROOT      = /mnt/Onetera/iss_group_24/dataset/aggregated
OUT_ROOT       = /mnt/Onetera/iss_group_24/checkpoints
ANALYSIS_ROOT  = /mnt/Onetera/iss_group_24/analysis


---
## Step 0 — Build dataset manifest (writes to Drive)

Stages HOTS + InsDet → `{DRIVE_ROOT}/dataset/aggregated/{train,test}`.
**Idempotent**: re-runs only when staging is missing or invalid. Use
`aggregator.main(force=True)` to rebuild.

Note: this writes to Drive even on Colab — Drive is the source of truth for
the staged dataset and we mirror it locally in the next cell for speed.


In [6]:
import aggregator
aggregator.main()

validate: checked 9083 (path, bbox) pairs, 0 bad
aggregator: dataset already staged at dataset/aggregated and validates; skipping.


---
## Step 0b — (Colab only) Mirror dataset to local runtime SSD

Reads `{DRIVE_ROOT}/dataset/aggregated/` and copies it to `LOCAL_DATA_ROOT`
on the Colab runtime, with progress printed. Drive random-read latency is
~50-100 ms per file; copying once and reading from local SSD takes the
per-file latency down to <1 ms, easily 50× faster training.

After the copy, `DATA_ROOT` and `MANIFEST` are re-pointed to the local copy.
**`OUT_ROOT` and `ANALYSIS_ROOT` continue to point to Drive** so checkpoints
and analysis JSONs persist across runtime resets.

If anything goes wrong (out of disk, permission, etc.) we fall back to reading
directly from Drive.


In [7]:
def mirror_to_local(src: str, dst: str, *, progress_every: float = 2.0) -> bool:
    """Copy a directory tree from src → dst with periodic progress prints.

    Returns True on success, False otherwise. Falls back to no-op when
    src and dst resolve to the same path.

    Skips files that already exist at dst with the same size + mtime, so
    re-running is cheap.
    """
    src_p = Path(src).resolve()
    dst_p = Path(dst).resolve()
    if src_p == dst_p:
        print(f"  src == dst ({src_p}); nothing to do.")
        return True
    if not src_p.exists():
        print(f"  ✗ source not found: {src_p}")
        return False

    # First pass: enumerate.
    files: list[tuple[Path, Path]] = []
    total_bytes = 0
    for p in src_p.rglob("*"):
        if p.is_file():
            rel = p.relative_to(src_p)
            target = dst_p / rel
            files.append((p, target))
            try:
                total_bytes += p.stat().st_size
            except OSError:
                pass
    n_files = len(files)
    print(f"  found {n_files} files ({total_bytes / 1024 / 1024:.1f} MB) under {src_p}")

    dst_p.mkdir(parents=True, exist_ok=True)
    n_copied = 0
    n_skipped = 0
    bytes_copied = 0
    t_start = time.time()
    t_last = t_start
    for s, d in files:
        try:
            d.parent.mkdir(parents=True, exist_ok=True)
            if d.exists():
                try:
                    s_st = s.stat()
                    d_st = d.stat()
                    if s_st.st_size == d_st.st_size and abs(s_st.st_mtime - d_st.st_mtime) < 1:
                        n_skipped += 1
                        n_copied += 1
                        bytes_copied += s_st.st_size
                        continue
                except OSError:
                    pass
            shutil.copy2(str(s), str(d))
            try:
                bytes_copied += s.stat().st_size
            except OSError:
                pass
            n_copied += 1
        except Exception as e:                                                # noqa: BLE001
            print(f"  ✗ failed to copy {s} → {d}: {e}")
            return False
        now = time.time()
        if now - t_last >= progress_every or n_copied == n_files:
            elapsed = now - t_start
            mb_done = bytes_copied / 1024 / 1024
            mb_total = total_bytes / 1024 / 1024
            rate = mb_done / max(elapsed, 1e-6)
            pct = 100.0 * n_copied / max(n_files, 1)
            eta = (mb_total - mb_done) / max(rate, 1e-6)
            print(
                f"  [{n_copied}/{n_files} = {pct:5.1f}%]  "
                f"{mb_done:7.1f} / {mb_total:7.1f} MB  "
                f"({rate:6.1f} MB/s)  elapsed={elapsed:5.1f}s  eta={eta:5.1f}s",
                flush=True,
            )
            t_last = now
    print(f"  ✓ copied {n_copied} files ({n_skipped} unchanged) in {time.time() - t_start:.1f}s")
    return True


if USE_GOOGLE_COLAB:
    print(f"Mirroring {DRIVE_DATA_ROOT} → {LOCAL_DATA_ROOT} …")
    ok = mirror_to_local(DRIVE_DATA_ROOT, LOCAL_DATA_ROOT)
    if ok:
        DATA_ROOT = LOCAL_DATA_ROOT
        MANIFEST  = f"{LOCAL_DATA_ROOT}/manifest.json"
        print(f"  ✓ DATA_ROOT switched to local: {DATA_ROOT}")
        print(f"  ✓ MANIFEST  switched to local: {MANIFEST}")
    else:
        print("  ⚠ mirror failed; falling back to Drive paths.")
        DATA_ROOT = DRIVE_DATA_ROOT
        MANIFEST  = DRIVE_MANIFEST_PATH
else:
    print("Not on Colab — DATA_ROOT stays at the project's local copy.")

print()
print(f"DATA_ROOT      = {DATA_ROOT}")
print(f"MANIFEST       = {MANIFEST}")
print(f"OUT_ROOT       = {OUT_ROOT}    (always Drive on Colab)")
print(f"ANALYSIS_ROOT  = {ANALYSIS_ROOT}    (always Drive on Colab)")


Not on Colab — DATA_ROOT stays at the project's local copy.

DATA_ROOT      = /mnt/Onetera/iss_group_24/dataset/aggregated
MANIFEST       = /mnt/Onetera/iss_group_24/dataset/aggregated/manifest.json
OUT_ROOT       = /mnt/Onetera/iss_group_24/checkpoints    (always Drive on Colab)
ANALYSIS_ROOT  = /mnt/Onetera/iss_group_24/analysis    (always Drive on Colab)


---
## Step 1 — Smoke test (~60s)

End-to-end check: aggregator validate → localizer L1/L2/L3 → siamese S1/S2 →
combined inference → plots → checkpoint roundtrip. Smoke artifacts go to
`{OUT_ROOT}/_smoke/` and `{ANALYSIS_ROOT}/_smoke/` and are wiped at the end.

Set `RUN_SMOKE_TEST = False` to skip. Recommended on first run.


In [8]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    from shared.smoke import smoke_test
    res = smoke_test(seconds_budget=60.0, manifest=MANIFEST, data_root=DATA_ROOT)
    print(f"smoke status={res['status']}  wall_clock={res['wall_clock_seconds']:.1f}s")

---
## Step 2 — Imports + shared kwargs

Every parameter in `SHARED_KWARGS` is overridable per-stage. The localizer and
siamese pull defaults from their own `DEFAULT_CFG`; entries below are the most
common knobs you might want to tune.

If a training cell is interrupted (Ctrl-C / kernel-interrupt), the trainer
catches the interrupt and frees CUDA VRAM before re-raising — no kernel
restart needed.


In [9]:
import torch
from localizer.train import (
    train_phase0    as loc_train_phase0,
    train_stage_L1  as loc_train_L1,
    train_stage_L2  as loc_train_L2,
    train_stage_L3  as loc_train_L3,
    evaluate_phase0 as loc_evaluate_phase0,
    evaluate_run    as loc_evaluate_run,
)
from siamese.train import (
    train_phase0    as sia_train_phase0,
    train_stage_S1  as sia_train_S1,
    train_stage_S2  as sia_train_S2,
    evaluate_phase0 as sia_evaluate_phase0,
    evaluate_run    as sia_evaluate_run,
)
from inference_combined import run_combined, sweep_threshold
from shared.plots import plot_all_from_jsons
from shared.runtime import release_gpu_memory

/mnt/Onetera/iss_group_24/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# VRAM-aware config tier.
_VRAM = torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else 0
_HAS_BIG_GPU = _VRAM >= 16e9
if _HAS_BIG_GPU:
    LOC_IMG, LOC_BATCH, LOC_ACCUM = 960, 4, 2
    SIA_IMG, SIA_BATCH, SIA_ACCUM = 518, 8, 1
elif _VRAM >= 5.5e9:
    LOC_IMG, LOC_BATCH, LOC_ACCUM = 768, 1, 8
    SIA_IMG, SIA_BATCH, SIA_ACCUM = 518, 4, 2
else:
    LOC_IMG, LOC_BATCH, LOC_ACCUM = 768, 1, 8
    SIA_IMG, SIA_BATCH, SIA_ACCUM = 518, 2, 4
print(f"VRAM={_VRAM/1e9:.1f} GB | localizer img={LOC_IMG} B={LOC_BATCH} accum={LOC_ACCUM} | siamese img={SIA_IMG} B={SIA_BATCH} accum={SIA_ACCUM}")

VRAM=6.0 GB | localizer img=768 B=1 accum=8 | siamese img=518 B=4 accum=2


In [11]:
# Shared paths/IO/folds — every key is forwardable to both trainers.
SHARED_KWARGS = dict(
    manifest      = MANIFEST,       # local SSD on Colab; Drive elsewhere
    data_root     = DATA_ROOT,      # local SSD on Colab; Drive elsewhere
    out_root      = OUT_ROOT,       # always Drive
    analysis_root = ANALYSIS_ROOT,  # always Drive
    num_workers   = 2,
    use_amp       = True,
    seed          = 42,
    folds         = 3,
    fold_seed     = 42,
    k_min         = 1,
    k_max         = 10,
    keep_last_n   = 0,             # keep every per-(epoch, fold) ckpt
)

# Localizer-specific overrides.
LOC_KWARGS = dict(
    **SHARED_KWARGS,
    img_size         = LOC_IMG,
    batch_size       = LOC_BATCH,
    grad_accum_steps = LOC_ACCUM,
    L1_epochs        = 3,
    L2_epochs        = 12,
    L3_epochs        = 12,
    L1_eps_per_fold  = 400,
    L2_eps_per_fold  = 250,
    L3_eps_per_fold  = 250,
    val_episodes     = 100,
    test_episodes    = 400,
    lr_fusion_L1     = 1e-4,
    lr_fusion_L2     = 1e-4,
    lr_class_L2      = 5e-5,
    lr_box_L2        = 5e-5,
    lr_fusion_L3     = 5e-5,
    lr_class_L3      = 2e-5,
    lr_box_L3        = 2e-5,
    lr_lora_L3       = 1e-4,
    lambda_patch_ce  = 1.0,
    lambda_l1        = 5.0,
    lambda_giou      = 2.0,
    patch_ce_label_smoothing = 0.0,
    L2_box_warmup_epochs = 2,
    fusion_layers    = 2,
    fusion_heads     = 8,
    fusion_dropout   = 0.0,
    lora_r           = 8,
    lora_alpha       = 16,
    lora_dropout     = 0.1,
    lora_last_n_layers = 4,
    L1_early_stop_patience = 4,
    L2_early_stop_patience = 4,
    L3_early_stop_patience = 4,
)

# Siamese-specific overrides.
SIA_KWARGS = dict(
    **SHARED_KWARGS,
    img_size         = SIA_IMG,
    batch_size       = SIA_BATCH,
    grad_accum_steps = SIA_ACCUM,
    S1_epochs        = 10,
    S2_epochs        = 8,
    S1_eps_per_fold  = 400,
    S2_eps_per_fold  = 400,
    val_episodes     = 200,
    test_episodes    = 400,
    lr_head_S1       = 1e-3,
    lr_head_S2       = 1e-4,
    lr_lora_S2       = 1e-4,
    focal_alpha      = 0.25,         # lower α ⇒ negatives weighted higher ⇒ FP penalty
    focal_gamma      = 2.0,
    variance_target  = 0.5,
    variance_weight  = 0.1,
    decorr_weight    = 0.05,
    neg_prob         = 0.75,         # 1:3 pos:neg
    hard_neg_cache_frac = 0.5,
    cross_attn_heads = 6,
    head_hidden_1    = 256,
    head_hidden_2    = 64,
    head_dropout     = 0.2,
    lora_r           = 8,
    lora_alpha       = 16,
    lora_dropout     = 0.1,
    lora_last_n_layers = 4,
    eval_threshold   = 0.5,
    early_stop_metric = "auroc",     # or "fpr_inv"
    S1_early_stop_patience = 4,
    S2_early_stop_patience = 4,
)

# Localizer

*OWLv2 + multi-shot fusion. Predicts bbox; positive-only training.*

## Localizer Phase 0 — zero-shot OWLv2

In [12]:
loc_train_phase0(**LOC_KWARGS)

=== [localizer] Phase 0 (zero-shot OWLv2) on cuda ===


Loading weights: 100%|██████████| 418/418 [00:00<00:00, 7201.17it/s]
/mnt/Onetera/iss_group_24/localizer/model.py:89: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.fusion = nn.TransformerEncoder(layer, num_layers=fusion_layers)


  evaluating: 400 batches
  [5/400]  elapsed= 12.4s  rate=0.40b/s
  [10/400]  elapsed= 23.1s  rate=0.43b/s
  [15/400]  elapsed= 33.8s  rate=0.44b/s
  [20/400]  elapsed= 44.6s  rate=0.45b/s
  [25/400]  elapsed= 55.4s  rate=0.45b/s
  [30/400]  elapsed= 66.2s  rate=0.45b/s
  [35/400]  elapsed= 77.0s  rate=0.45b/s
  [40/400]  elapsed= 87.8s  rate=0.46b/s
  [45/400]  elapsed= 98.7s  rate=0.46b/s
  [50/400]  elapsed=109.5s  rate=0.46b/s
  [55/400]  elapsed=120.4s  rate=0.46b/s
  [60/400]  elapsed=131.3s  rate=0.46b/s
  [65/400]  elapsed=142.2s  rate=0.46b/s
  [70/400]  elapsed=153.1s  rate=0.46b/s
  [75/400]  elapsed=164.0s  rate=0.46b/s
  [80/400]  elapsed=175.0s  rate=0.46b/s
  [85/400]  elapsed=185.9s  rate=0.46b/s
  [90/400]  elapsed=196.8s  rate=0.46b/s
  [95/400]  elapsed=207.8s  rate=0.46b/s
  [100/400]  elapsed=218.8s  rate=0.46b/s
  [105/400]  elapsed=229.8s  rate=0.46b/s
  [110/400]  elapsed=240.8s  rate=0.46b/s
  [115/400]  elapsed=251.8s  rate=0.46b/s
  [120/400]  elapsed=262.8s 

{'test': {'overall': {'n': 400,
   'n_pos': 400,
   'iou_mean': 0.03484087570736847,
   'iou_median': 0.0004091168084414676,
   'iou_p25': 0.0,
   'iou_p75': 0.002688045962713659,
   'iou_std': 0.13939474946438482,
   'frac_iou_25': 0.04,
   'frac_iou_50': 0.035,
   'frac_iou_75': 0.01,
   'frac_iou_90': 0.0,
   'containment_mean': 0.5953426789212972,
   'containment_median': 0.9953235387802124,
   'containment_p25': 0.0,
   'containment_p75': 0.9994310587644577,
   'containment_std': 0.48493740060763224,
   'frac_containment_50': 0.6,
   'frac_containment_75': 0.59,
   'frac_containment_90': 0.5825,
   'frac_containment_full': 0.5525,
   'contain_at_iou_50': 0.035,
   'contain_at_iou_75': 0.01,
   'high_contain_high_iou': 0.0325,
   'ap_per_iou': {'0.50': 0.004501142421934502,
    '0.55': 0.004501142421934502,
    '0.60': 0.0015284166211896781,
    '0.65': 0.0015284166211896781,
    '0.70': 0.0014789116706946286,
    '0.75': 0.0002292860865033872,
    '0.80': 0.00013027618551328816,
 

### Evaluate Phase 0 on test split

In [13]:
loc_evaluate_phase0(**LOC_KWARGS)

=== [localizer] Phase 0 evaluation on test split (cuda) ===


Loading weights: 100%|██████████| 418/418 [00:00<00:00, 5723.38it/s]


  evaluating: 400 batches
  [5/400]  elapsed= 11.6s  rate=0.43b/s
  [10/400]  elapsed= 22.6s  rate=0.44b/s
  [15/400]  elapsed= 33.5s  rate=0.45b/s
  [20/400]  elapsed= 44.5s  rate=0.45b/s
  [25/400]  elapsed= 55.5s  rate=0.45b/s
  [30/400]  elapsed= 66.5s  rate=0.45b/s
  [35/400]  elapsed= 77.5s  rate=0.45b/s
  [40/400]  elapsed= 88.5s  rate=0.45b/s
  [45/400]  elapsed= 99.5s  rate=0.45b/s
  [50/400]  elapsed=110.4s  rate=0.45b/s
  [55/400]  elapsed=121.4s  rate=0.45b/s
  [60/400]  elapsed=132.4s  rate=0.45b/s
  [65/400]  elapsed=143.3s  rate=0.45b/s
  [70/400]  elapsed=154.2s  rate=0.45b/s
  [75/400]  elapsed=165.2s  rate=0.45b/s
  [80/400]  elapsed=176.2s  rate=0.45b/s
  [85/400]  elapsed=187.1s  rate=0.45b/s
  [90/400]  elapsed=198.1s  rate=0.45b/s
  [95/400]  elapsed=209.0s  rate=0.45b/s
  [100/400]  elapsed=220.0s  rate=0.45b/s
  [105/400]  elapsed=230.9s  rate=0.45b/s
  [110/400]  elapsed=241.9s  rate=0.45b/s
  [115/400]  elapsed=252.8s  rate=0.45b/s
  [120/400]  elapsed=263.8s 

{'overall': {'n': 400,
  'n_pos': 400,
  'iou_mean': 0.03484087570736847,
  'iou_median': 0.0004091168084414676,
  'iou_p25': 0.0,
  'iou_p75': 0.002688045962713659,
  'iou_std': 0.13939474946438482,
  'frac_iou_25': 0.04,
  'frac_iou_50': 0.035,
  'frac_iou_75': 0.01,
  'frac_iou_90': 0.0,
  'containment_mean': 0.5953426789212972,
  'containment_median': 0.9953235387802124,
  'containment_p25': 0.0,
  'containment_p75': 0.9994310587644577,
  'containment_std': 0.48493740060763224,
  'frac_containment_50': 0.6,
  'frac_containment_75': 0.59,
  'frac_containment_90': 0.5825,
  'frac_containment_full': 0.5525,
  'contain_at_iou_50': 0.035,
  'contain_at_iou_75': 0.01,
  'high_contain_high_iou': 0.0325,
  'ap_per_iou': {'0.50': 0.004501142421934502,
   '0.55': 0.004501142421934502,
   '0.60': 0.0015284166211896781,
   '0.65': 0.0015284166211896781,
   '0.70': 0.0014789116706946286,
   '0.75': 0.0002292860865033872,
   '0.80': 0.00013027618551328816,
   '0.85': 0.0,
   '0.90': 0.0,
   '0.9

## Localizer Stage L1 — fusion transformer only

Freezes OWLv2 entirely; trains only the [CLS]+transformer fusion.

In [14]:
loc_train_L1(**LOC_KWARGS)


=== [localizer] Stage L1 on cuda ===


Loading weights: 100%|██████████| 418/418 [00:00<00:00, 3787.39it/s]


▶ [localizer] L1 epoch 1/3 fold 0/2
  training : 400 batches  (box_loss off)
  [10/400=  2.5%]  elapsed=  8.2s  eta=321.5s  rate=1.21b/s  loss=8.1492  patch_ce=8.1492
  [20/400=  5.0%]  elapsed= 14.9s  eta=282.2s  rate=1.35b/s  loss=7.6553  patch_ce=7.6553
  [30/400=  7.5%]  elapsed= 21.2s  eta=261.1s  rate=1.42b/s  loss=7.6617  patch_ce=7.6617
  [40/400= 10.0%]  elapsed= 27.5s  eta=247.1s  rate=1.46b/s  loss=7.6339  patch_ce=7.6339
  [50/400= 12.5%]  elapsed= 33.7s  eta=236.0s  rate=1.48b/s  loss=8.0165  patch_ce=8.0165
  [60/400= 15.0%]  elapsed= 39.9s  eta=226.3s  rate=1.50b/s  loss=7.9579  patch_ce=7.9579
  [70/400= 17.5%]  elapsed= 46.3s  eta=218.3s  rate=1.51b/s  loss=7.8890  patch_ce=7.8890
  [80/400= 20.0%]  elapsed= 52.5s  eta=210.1s  rate=1.52b/s  loss=7.8795  patch_ce=7.8795
  [90/400= 22.5%]  elapsed= 59.1s  eta=203.4s  rate=1.52b/s  loss=7.7862  patch_ce=7.7862
  [100/400= 25.0%]  elapsed= 66.0s  eta=197.9s  rate=1.52b/s  loss=7.7063  patch_ce=7.7063
  [110/400= 27.5%]  el

{'best_metric': {'value': 0.09018739558985277, 'epoch': 3, 'fold': 2},
 'config': {'manifest': '/mnt/Onetera/iss_group_24/dataset/aggregated/manifest.json',
  'data_root': '/mnt/Onetera/iss_group_24/dataset/aggregated',
  'out_root': '/mnt/Onetera/iss_group_24/checkpoints',
  'analysis_root': '/mnt/Onetera/iss_group_24/analysis',
  'img_size': 768,
  'batch_size': 1,
  'grad_accum_steps': 8,
  'num_workers': 2,
  'use_amp': True,
  'device': None,
  'folds': 3,
  'fold_seed': 42,
  'k_min': 1,
  'k_max': 10,
  'L1_epochs': 3,
  'L2_epochs': 12,
  'L3_epochs': 8,
  'L1_eps_per_fold': 400,
  'L2_eps_per_fold': 250,
  'L3_eps_per_fold': 250,
  'val_episodes': 100,
  'test_episodes': 400,
  'lr_fusion_L1': 0.0001,
  'lr_fusion_L2': 0.0001,
  'lr_class_L2': 5e-05,
  'lr_box_L2': 5e-05,
  'lr_fusion_L3': 5e-05,
  'lr_class_L3': 2e-05,
  'lr_box_L3': 2e-05,
  'lr_lora_L3': 0.0001,
  'weight_decay': 0.0001,
  'grad_clip': 1.0,
  'warmup_frac': 0.05,
  'lambda_patch_ce': 1.0,
  'lambda_l1': 5.0

In [15]:
loc_evaluate_run(checkpoint=f'{OUT_ROOT}/localizer/L1/stage_complete.pt', **LOC_KWARGS)

=== [localizer] Evaluating /mnt/Onetera/iss_group_24/checkpoints/localizer/L1/stage_complete.pt on test split (cuda) ===


Loading weights: 100%|██████████| 418/418 [00:00<00:00, 2269.82it/s]


  evaluating: 400 batches
  [5/400]  elapsed= 11.4s  rate=0.44b/s
  [10/400]  elapsed= 22.3s  rate=0.45b/s
  [15/400]  elapsed= 33.3s  rate=0.45b/s
  [20/400]  elapsed= 44.2s  rate=0.45b/s
  [25/400]  elapsed= 55.1s  rate=0.45b/s
  [30/400]  elapsed= 66.0s  rate=0.45b/s
  [35/400]  elapsed= 77.0s  rate=0.45b/s
  [40/400]  elapsed= 87.9s  rate=0.45b/s
  [45/400]  elapsed= 98.9s  rate=0.46b/s
  [50/400]  elapsed=109.8s  rate=0.46b/s
  [55/400]  elapsed=120.8s  rate=0.46b/s
  [60/400]  elapsed=131.7s  rate=0.46b/s
  [65/400]  elapsed=142.6s  rate=0.46b/s
  [70/400]  elapsed=153.5s  rate=0.46b/s
  [75/400]  elapsed=164.5s  rate=0.46b/s
  [80/400]  elapsed=175.4s  rate=0.46b/s
  [85/400]  elapsed=186.4s  rate=0.46b/s
  [90/400]  elapsed=197.3s  rate=0.46b/s
  [95/400]  elapsed=208.2s  rate=0.46b/s
  [100/400]  elapsed=219.2s  rate=0.46b/s
  [105/400]  elapsed=230.2s  rate=0.46b/s
  [110/400]  elapsed=241.1s  rate=0.46b/s
  [115/400]  elapsed=252.0s  rate=0.46b/s
  [120/400]  elapsed=262.9s 

{'overall': {'n': 400,
  'n_pos': 400,
  'iou_mean': 0.08821865618374432,
  'iou_median': 0.0,
  'iou_p25': 0.0,
  'iou_p75': 0.0036804850678890944,
  'iou_std': 0.2254574529691729,
  'frac_iou_25': 0.13,
  'frac_iou_50': 0.105,
  'frac_iou_75': 0.03,
  'frac_iou_90': 0.0,
  'containment_mean': 0.3598835127661005,
  'containment_median': 0.0,
  'containment_p25': 0.0,
  'containment_p75': 0.9956533312797546,
  'containment_std': 0.4653542557913728,
  'frac_containment_50': 0.3675,
  'frac_containment_75': 0.35,
  'frac_containment_90': 0.315,
  'frac_containment_full': 0.2825,
  'contain_at_iou_50': 0.105,
  'contain_at_iou_75': 0.03,
  'high_contain_high_iou': 0.07,
  'ap_per_iou': {'0.50': 0.023745261514791553,
   '0.55': 0.023371374388184433,
   '0.60': 0.020388821787767686,
   '0.65': 0.019734606898946412,
   '0.70': 0.0176852514518784,
   '0.75': 0.0024751600221565567,
   '0.80': 0.0007976687112992393,
   '0.85': 0.0,
   '0.90': 0.0,
   '0.95': 0.0},
  'map_50': 0.0237452615147915

## Localizer Stage L2 — + class_head + box_head

Unfreezes OWLv2's class_head + box_head + layer_norm. Box head is frozen for the first 2 epochs (configurable via `L2_box_warmup_epochs`).

In [16]:
loc_train_L2(**LOC_KWARGS)


=== [localizer] Stage L2 on cuda ===


Loading weights: 100%|██████████| 418/418 [00:00<00:00, 3410.42it/s]


  warm-starting from /mnt/Onetera/iss_group_24/checkpoints/localizer/L1/stage_complete.pt
▶ [localizer] L2 epoch 1/12 fold 0/2
  training : 250 batches  (box_loss on)
  [10/250=  4.0%]  elapsed=  9.5s  eta=228.7s  rate=1.05b/s  loss=7.7720  patch_ce=4.2199
  [20/250=  8.0%]  elapsed= 15.8s  eta=182.1s  rate=1.26b/s  loss=7.9384  patch_ce=4.4735
  [30/250= 12.0%]  elapsed= 22.5s  eta=165.3s  rate=1.33b/s  loss=7.9643  patch_ce=4.5096
  [40/250= 16.0%]  elapsed= 29.1s  eta=152.7s  rate=1.38b/s  loss=7.7788  patch_ce=4.5185
  [50/250= 20.0%]  elapsed= 36.6s  eta=146.2s  rate=1.37b/s  loss=7.8214  patch_ce=4.5113
  [60/250= 24.0%]  elapsed= 44.5s  eta=141.1s  rate=1.35b/s  loss=7.7871  patch_ce=4.4935
  [70/250= 28.0%]  elapsed= 51.0s  eta=131.1s  rate=1.37b/s  loss=7.8926  patch_ce=4.5680
  [80/250= 32.0%]  elapsed= 57.4s  eta=121.9s  rate=1.39b/s  loss=7.8125  patch_ce=4.4768
  [90/250= 36.0%]  elapsed= 64.5s  eta=114.7s  rate=1.40b/s  loss=7.7736  patch_ce=4.4995
  [100/250= 40.0%]  ela

{'best_metric': {'value': 0.4412661490877284, 'epoch': 10, 'fold': 2},
 'config': {'manifest': '/mnt/Onetera/iss_group_24/dataset/aggregated/manifest.json',
  'data_root': '/mnt/Onetera/iss_group_24/dataset/aggregated',
  'out_root': '/mnt/Onetera/iss_group_24/checkpoints',
  'analysis_root': '/mnt/Onetera/iss_group_24/analysis',
  'img_size': 768,
  'batch_size': 1,
  'grad_accum_steps': 8,
  'num_workers': 2,
  'use_amp': True,
  'device': None,
  'folds': 3,
  'fold_seed': 42,
  'k_min': 1,
  'k_max': 10,
  'L1_epochs': 3,
  'L2_epochs': 12,
  'L3_epochs': 8,
  'L1_eps_per_fold': 400,
  'L2_eps_per_fold': 250,
  'L3_eps_per_fold': 250,
  'val_episodes': 100,
  'test_episodes': 400,
  'lr_fusion_L1': 0.0001,
  'lr_fusion_L2': 0.0001,
  'lr_class_L2': 5e-05,
  'lr_box_L2': 5e-05,
  'lr_fusion_L3': 5e-05,
  'lr_class_L3': 2e-05,
  'lr_box_L3': 2e-05,
  'lr_lora_L3': 0.0001,
  'weight_decay': 0.0001,
  'grad_clip': 1.0,
  'warmup_frac': 0.05,
  'lambda_patch_ce': 1.0,
  'lambda_l1': 5.0

In [17]:
loc_evaluate_run(checkpoint=f'{OUT_ROOT}/localizer/L2/stage_complete.pt', **LOC_KWARGS)

=== [localizer] Evaluating /mnt/Onetera/iss_group_24/checkpoints/localizer/L2/stage_complete.pt on test split (cuda) ===


Loading weights: 100%|██████████| 418/418 [00:00<00:00, 2242.29it/s]


  evaluating: 400 batches
  [5/400]  elapsed= 11.5s  rate=0.43b/s
  [10/400]  elapsed= 22.6s  rate=0.44b/s
  [15/400]  elapsed= 33.7s  rate=0.45b/s
  [20/400]  elapsed= 44.8s  rate=0.45b/s
  [25/400]  elapsed= 55.9s  rate=0.45b/s
  [30/400]  elapsed= 67.0s  rate=0.45b/s
  [35/400]  elapsed= 78.0s  rate=0.45b/s
  [40/400]  elapsed= 89.1s  rate=0.45b/s
  [45/400]  elapsed=100.2s  rate=0.45b/s
  [50/400]  elapsed=111.3s  rate=0.45b/s
  [55/400]  elapsed=122.4s  rate=0.45b/s
  [60/400]  elapsed=133.5s  rate=0.45b/s
  [65/400]  elapsed=144.6s  rate=0.45b/s
  [70/400]  elapsed=155.7s  rate=0.45b/s
  [75/400]  elapsed=166.8s  rate=0.45b/s
  [80/400]  elapsed=177.9s  rate=0.45b/s
  [85/400]  elapsed=189.0s  rate=0.45b/s
  [90/400]  elapsed=200.1s  rate=0.45b/s
  [95/400]  elapsed=211.2s  rate=0.45b/s
  [100/400]  elapsed=222.3s  rate=0.45b/s
  [105/400]  elapsed=233.5s  rate=0.45b/s
  [110/400]  elapsed=244.6s  rate=0.45b/s
  [115/400]  elapsed=255.7s  rate=0.45b/s
  [120/400]  elapsed=266.8s 

{'overall': {'n': 400,
  'n_pos': 400,
  'iou_mean': 0.1965969323826721,
  'iou_median': 0.0,
  'iou_p25': 0.0,
  'iou_p75': 0.34682750701904297,
  'iou_std': 0.3246356399807602,
  'frac_iou_25': 0.2675,
  'frac_iou_50': 0.215,
  'frac_iou_75': 0.14,
  'frac_iou_90': 0.04,
  'containment_mean': 0.24262357854284347,
  'containment_median': 0.0,
  'containment_p25': 0.0,
  'containment_p75': 0.4995122253894806,
  'containment_std': 0.3839525639082449,
  'frac_containment_50': 0.2425,
  'frac_containment_75': 0.205,
  'frac_containment_90': 0.1575,
  'frac_containment_full': 0.0775,
  'contain_at_iou_50': 0.215,
  'contain_at_iou_75': 0.14,
  'high_contain_high_iou': 0.1425,
  'ap_per_iou': {'0.50': 0.09113205476912783,
   '0.55': 0.07902856388406743,
   '0.60': 0.07526552165000763,
   '0.65': 0.06833161022772999,
   '0.70': 0.0652038508178362,
   '0.75': 0.051913847536477584,
   '0.80': 0.04176173115716015,
   '0.85': 0.02502777090953451,
   '0.90': 0.003442980298230248,
   '0.95': 0.000

## Localizer Stage L3 — + LoRA on last 4 ViT blocks

In [12]:
loc_train_L3(**LOC_KWARGS)


=== [localizer] Stage L3 on cuda ===


Loading weights: 100%|██████████| 418/418 [00:00<00:00, 6427.09it/s]
/mnt/Onetera/iss_group_24/localizer/model.py:89: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.fusion = nn.TransformerEncoder(layer, num_layers=fusion_layers)


  resuming from /mnt/Onetera/iss_group_24/checkpoints/localizer/L3/last.pt
  restored optim+sched at epoch=7 fold=0; continuing from epoch=7 fold=1
▶ [localizer] L3 epoch 7/12 fold 1/2
  training : 250 batches  (box_loss on)
  [10/250=  4.0%]  elapsed= 12.8s  eta=308.0s  rate=0.78b/s  loss=3.8007  patch_ce=2.4465
  [20/250=  8.0%]  elapsed= 22.8s  eta=262.3s  rate=0.88b/s  loss=4.8981  patch_ce=3.1711
  [30/250= 12.0%]  elapsed= 32.8s  eta=240.5s  rate=0.91b/s  loss=4.9973  patch_ce=3.1207
  [40/250= 16.0%]  elapsed= 42.9s  eta=225.0s  rate=0.93b/s  loss=4.7413  patch_ce=3.1155
  [50/250= 20.0%]  elapsed= 53.0s  eta=211.9s  rate=0.94b/s  loss=5.0023  patch_ce=3.1088
  [60/250= 24.0%]  elapsed= 63.1s  eta=199.8s  rate=0.95b/s  loss=5.0610  patch_ce=3.1788
  [70/250= 28.0%]  elapsed= 73.1s  eta=188.1s  rate=0.96b/s  loss=5.2070  patch_ce=3.2828
  [80/250= 32.0%]  elapsed= 83.2s  eta=176.8s  rate=0.96b/s  loss=5.2730  patch_ce=3.4221
  [90/250= 36.0%]  elapsed= 93.4s  eta=166.0s  rate=0.9

{'best_metric': {'value': 0.3841712191138737, 'epoch': 4, 'fold': 2},
 'config': {'manifest': '/mnt/Onetera/iss_group_24/dataset/aggregated/manifest.json',
  'data_root': '/mnt/Onetera/iss_group_24/dataset/aggregated',
  'out_root': '/mnt/Onetera/iss_group_24/checkpoints',
  'analysis_root': '/mnt/Onetera/iss_group_24/analysis',
  'img_size': 768,
  'batch_size': 1,
  'grad_accum_steps': 8,
  'num_workers': 2,
  'use_amp': True,
  'device': None,
  'folds': 3,
  'fold_seed': 42,
  'k_min': 1,
  'k_max': 10,
  'L1_epochs': 3,
  'L2_epochs': 12,
  'L3_epochs': 12,
  'L1_eps_per_fold': 400,
  'L2_eps_per_fold': 250,
  'L3_eps_per_fold': 250,
  'val_episodes': 100,
  'test_episodes': 400,
  'lr_fusion_L1': 0.0001,
  'lr_fusion_L2': 0.0001,
  'lr_class_L2': 5e-05,
  'lr_box_L2': 5e-05,
  'lr_fusion_L3': 5e-05,
  'lr_class_L3': 2e-05,
  'lr_box_L3': 2e-05,
  'lr_lora_L3': 0.0001,
  'weight_decay': 0.0001,
  'grad_clip': 1.0,
  'warmup_frac': 0.05,
  'lambda_patch_ce': 1.0,
  'lambda_l1': 5.0

In [13]:
loc_evaluate_run(checkpoint=f'{OUT_ROOT}/localizer/L3/stage_complete.pt', **LOC_KWARGS)

=== [localizer] Evaluating /mnt/Onetera/iss_group_24/checkpoints/localizer/L3/stage_complete.pt on test split (cuda) ===


Loading weights: 100%|██████████| 418/418 [00:00<00:00, 4992.71it/s]


  evaluating: 400 batches
  [5/400]  elapsed= 11.0s  rate=0.45b/s
  [10/400]  elapsed= 21.7s  rate=0.46b/s
  [15/400]  elapsed= 32.4s  rate=0.46b/s
  [20/400]  elapsed= 43.2s  rate=0.46b/s
  [25/400]  elapsed= 54.0s  rate=0.46b/s
  [30/400]  elapsed= 64.9s  rate=0.46b/s
  [35/400]  elapsed= 75.8s  rate=0.46b/s
  [40/400]  elapsed= 86.7s  rate=0.46b/s
  [45/400]  elapsed= 97.6s  rate=0.46b/s
  [50/400]  elapsed=108.5s  rate=0.46b/s
  [55/400]  elapsed=119.5s  rate=0.46b/s
  [60/400]  elapsed=130.4s  rate=0.46b/s
  [65/400]  elapsed=141.4s  rate=0.46b/s
  [70/400]  elapsed=152.5s  rate=0.46b/s
  [75/400]  elapsed=163.5s  rate=0.46b/s
  [80/400]  elapsed=174.6s  rate=0.46b/s
  [85/400]  elapsed=185.7s  rate=0.46b/s
  [90/400]  elapsed=196.8s  rate=0.46b/s
  [95/400]  elapsed=207.8s  rate=0.46b/s
  [100/400]  elapsed=219.0s  rate=0.46b/s
  [105/400]  elapsed=230.1s  rate=0.46b/s
  [110/400]  elapsed=241.3s  rate=0.46b/s
  [115/400]  elapsed=252.4s  rate=0.46b/s
  [120/400]  elapsed=263.5s 

{'overall': {'n': 400,
  'n_pos': 400,
  'iou_mean': 0.19347335477617889,
  'iou_median': 0.0,
  'iou_p25': 0.0,
  'iou_p75': 0.1786128245294094,
  'iou_std': 0.3367904287817796,
  'frac_iou_25': 0.2375,
  'frac_iou_50': 0.1975,
  'frac_iou_75': 0.16,
  'frac_iou_90': 0.0775,
  'containment_mean': 0.2200443089879991,
  'containment_median': 0.0,
  'containment_p25': 0.0,
  'containment_p75': 0.2513071298599243,
  'containment_std': 0.3710986460425294,
  'frac_containment_50': 0.2125,
  'frac_containment_75': 0.1825,
  'frac_containment_90': 0.1425,
  'frac_containment_full': 0.0275,
  'contain_at_iou_50': 0.1975,
  'contain_at_iou_75': 0.16,
  'high_contain_high_iou': 0.13,
  'ap_per_iou': {'0.50': 0.07366246344825852,
   '0.55': 0.07276338930281262,
   '0.60': 0.07276338930281262,
   '0.65': 0.06674941782173059,
   '0.70': 0.06304702794953344,
   '0.75': 0.054131854270719156,
   '0.80': 0.04922867754139209,
   '0.85': 0.04395777463561939,
   '0.90': 0.020875338529145623,
   '0.95': 0.

In [ ]:
loc_evaluate_run(checkpoint=f'{OUT_ROOT}/localizer/L3/best.pt', **LOC_KWARGS)

In [ ]:
loc_evaluate_run(checkpoint=f'{OUT_ROOT}/localizer/L2/best.pt', **LOC_KWARGS)

---
# Siamese

*DINOv2-small + cross-attention pooling head. Predicts existence_prob; mixed pos:neg = 1:3.*

## Siamese Phase 0 — zero-shot DINOv2 cosine baseline

In [14]:
sia_train_phase0(**SIA_KWARGS)

=== [siamese] Phase 0 evaluation (zero-shot DINOv2 cosine) on cuda ===


Loading weights: 100%|██████████| 223/223 [00:00<00:00, 2935.04it/s]


  evaluating: 100 batches
  [5/100]  elapsed= 10.0s  rate=0.50b/s
  [10/100]  elapsed= 17.4s  rate=0.57b/s
  [15/100]  elapsed= 24.8s  rate=0.60b/s
  [20/100]  elapsed= 32.5s  rate=0.62b/s
  [25/100]  elapsed= 39.5s  rate=0.63b/s
  [30/100]  elapsed= 47.4s  rate=0.63b/s
  [35/100]  elapsed= 54.5s  rate=0.64b/s
  [40/100]  elapsed= 61.4s  rate=0.65b/s
  [45/100]  elapsed= 68.6s  rate=0.66b/s
  [50/100]  elapsed= 76.0s  rate=0.66b/s
  [55/100]  elapsed= 82.8s  rate=0.66b/s
  [60/100]  elapsed= 90.9s  rate=0.66b/s
  [65/100]  elapsed= 97.8s  rate=0.66b/s
  [70/100]  elapsed=104.7s  rate=0.67b/s
  [75/100]  elapsed=112.1s  rate=0.67b/s
  [80/100]  elapsed=119.2s  rate=0.67b/s
  [85/100]  elapsed=126.1s  rate=0.67b/s
  [90/100]  elapsed=133.9s  rate=0.67b/s
  [95/100]  elapsed=140.9s  rate=0.67b/s
  [100/100]  elapsed=147.9s  rate=0.68b/s
[siamese phase0] test  AUROC=0.6468  PR-AUC=0.3263  AP=0.3483  acc=0.3200  best_f1=0.4542@thr=0.55  FPR=0.8697  FNR=0.0538  MCC=0.1023  (147.9s)
[siamese]

{'overall': {'n': 400,
  'n_pos': 93,
  'n_neg': 307,
  'threshold': 0.5,
  'tp': 88,
  'fp': 267,
  'fn': 5,
  'tn': 40,
  'accuracy': 0.32,
  'acc_pos': 0.946236559139785,
  'acc_neg': 0.13029315960912052,
  'precision': 0.24788732394366197,
  'recall': 0.946236559139785,
  'f1': 0.39285714285714285,
  'fpr': 0.8697068403908795,
  'fnr': 0.053763440860215055,
  'tpr': 0.946236559139785,
  'tnr': 0.13029315960912047,
  'youden_j': 0.07652971874890546,
  'mcc': 0.10231053269148564,
  'auroc': 0.6467724422962419,
  'pr_auc': 0.3263125685446292,
  'avg_precision': 0.3482887463768518,
  'brier': 0.2806780893966687,
  'best_f1': 0.4542124542124542,
  'best_f1_threshold': 0.548857569694519,
  'fpr_at_recall_95': 0.8925081433224755,
  'recall_at_fpr_05': 0.08602150537634409,
  'recall_at_fpr_10': 0.11827956989247312,
  'mean_score_pos': 0.5782256645541037,
  'mean_score_neg': 0.5533697197413988,
  'std_score_pos': 0.06801328577922801,
  'std_score_neg': 0.06499414985922937,
  'score_gap': 0.

### Evaluate Phase 0 on test split

In [15]:
sia_evaluate_phase0(**SIA_KWARGS)

=== [siamese] Phase 0 evaluation (zero-shot DINOv2 cosine) on cuda ===


Loading weights: 100%|██████████| 223/223 [00:00<00:00, 4100.58it/s]

  evaluating: 100 batches


  [5/100]  elapsed=  9.8s  rate=0.51b/s
  [10/100]  elapsed= 17.4s  rate=0.57b/s
  [15/100]  elapsed= 24.4s  rate=0.61b/s
  [20/100]  elapsed= 31.4s  rate=0.64b/s
  [25/100]  elapsed= 38.3s  rate=0.65b/s
  [30/100]  elapsed= 45.8s  rate=0.66b/s
  [35/100]  elapsed= 52.8s  rate=0.66b/s
  [40/100]  elapsed= 59.7s  rate=0.67b/s
  [45/100]  elapsed= 66.7s  rate=0.67b/s
  [50/100]  elapsed= 73.9s  rate=0.68b/s
  [55/100]  elapsed= 80.9s  rate=0.68b/s
  [60/100]  elapsed= 88.5s  rate=0.68b/s
  [65/100]  elapsed= 95.5s  rate=0.68b/s
  [70/100]  elapsed=102.5s  rate=0.68b/s
  [75/100]  elapsed=109.5s  rate=0.69b/s
  [80/100]  elapsed=116.5s  rate=0.69b/s
  [85/100]  elapsed=123.5s  rate=0.69b/s
  [90/100]  elapsed=130.7s  rate=0.69b/s
  [95/100]  elapsed=137.7s  rate=0.69b/s
  [100/100]  elapsed=144.8s  rate=0.69b/s
[siamese phase0] test  AUROC=0.6468  PR-AUC=0.3263  AP=0.3483  acc=0.3200  best_f1=0.4542@thr=0.55  FPR=0.8697  FNR=0.0538  MCC=0.1023  (144.8s)
[siamese] Phase 0 complete. Results

{'overall': {'n': 400,
  'n_pos': 93,
  'n_neg': 307,
  'threshold': 0.5,
  'tp': 88,
  'fp': 267,
  'fn': 5,
  'tn': 40,
  'accuracy': 0.32,
  'acc_pos': 0.946236559139785,
  'acc_neg': 0.13029315960912052,
  'precision': 0.24788732394366197,
  'recall': 0.946236559139785,
  'f1': 0.39285714285714285,
  'fpr': 0.8697068403908795,
  'fnr': 0.053763440860215055,
  'tpr': 0.946236559139785,
  'tnr': 0.13029315960912047,
  'youden_j': 0.07652971874890546,
  'mcc': 0.10231053269148564,
  'auroc': 0.6467724422962419,
  'pr_auc': 0.3263125685446292,
  'avg_precision': 0.3482887463768518,
  'brier': 0.2806780893966687,
  'best_f1': 0.4542124542124542,
  'best_f1_threshold': 0.548857569694519,
  'fpr_at_recall_95': 0.8925081433224755,
  'recall_at_fpr_05': 0.08602150537634409,
  'recall_at_fpr_10': 0.11827956989247312,
  'mean_score_pos': 0.5782256645541037,
  'mean_score_neg': 0.5533697197413988,
  'std_score_pos': 0.06801328577922801,
  'std_score_neg': 0.06499414985922937,
  'score_gap': 0.

## Siamese Stage S1 — cross-attn head only

In [16]:
sia_train_S1(**SIA_KWARGS)


=== [siamese] Stage S1 on cuda ===


Loading weights: 100%|██████████| 223/223 [00:00<00:00, 7033.40it/s]

▶ [siamese] S1 epoch 1/10 fold 0/2
  training : 100 batches



💥 training raised an exception; releasing GPU VRAM …
  release_gpu_memory: allocated 4495 → 4495 MB, reserved 5018 → 5018 MB


OutOfMemoryError: CUDA out of memory. Tried to allocate 858.00 MiB. GPU 0 has a total capacity of 5.60 GiB of which 587.06 MiB is free. Including non-PyTorch memory, this process has 5.02 GiB memory in use. Of the allocated memory 4.39 GiB is allocated by PyTorch, and 523.39 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
sia_evaluate_run(checkpoint=f'{OUT_ROOT}/siamese/S1/stage_complete.pt', **SIA_KWARGS)

## Siamese Stage S2 — + LoRA on last 4 DINOv2 blocks

In [ ]:
sia_train_S2(**SIA_KWARGS)

In [ ]:
sia_evaluate_run(checkpoint=f'{OUT_ROOT}/siamese/S2/stage_complete.pt', **SIA_KWARGS)

---
# Combined cascaded inference

The two models are independent. To deploy: run siamese first, threshold its
`existence_prob`, and (in `"hard"` mode) only invoke the localizer if the
threshold passes. Threshold can be tuned post-hoc with `sweep_threshold`.


## Threshold sweep on the test split

In [ ]:
sweep_threshold(
    siamese_ckpt   = f'{OUT_ROOT}/siamese/S2/stage_complete.pt',
    localizer_ckpt = f'{OUT_ROOT}/localizer/L3/stage_complete.pt',
    manifest       = MANIFEST,
    data_root      = DATA_ROOT,
    test_episodes  = 400,
    thresholds     = (0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70),
    siamese_img_size   = SIA_IMG,
    localizer_img_size = LOC_IMG,
    analysis_root  = ANALYSIS_ROOT,
)

## Single-pair combined inference example

Replace paths with your own files. `existence_threshold_mode` options:
- `"hard"` (default): siamese gates the localizer
- `"soft"`: always run both, return both fields
- `"always_localize"`: always return bbox + existence_prob


In [ ]:
# Example — uncomment and replace paths.
# run_combined(
#     siamese_ckpt   = f'{OUT_ROOT}/siamese/S2/stage_complete.pt',
#     localizer_ckpt = f'{OUT_ROOT}/localizer/L3/stage_complete.pt',
#     support_paths  = ['s1.jpg', 's2.jpg', 's3.jpg', 's4.jpg'],
#     query_path     = 'scene.jpg',
#     existence_threshold      = 0.5,
#     existence_threshold_mode = 'hard',
#     siamese_img_size   = SIA_IMG,
#     localizer_img_size = LOC_IMG,
# )

---
# Plots

In [ ]:
plot_all_from_jsons(ANALYSIS_ROOT)

## Free GPU VRAM

If you ran a training cell and want to release GPU memory without
restarting the kernel (e.g. before opening a different notebook), call
`release_gpu_memory()`.

In [17]:
release_gpu_memory()

  release_gpu_memory: allocated 4495 → 4495 MB, reserved 5018 → 5018 MB


In [18]:
!nvidia-smi

Tue May 12 15:14:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2060        Off |   00000000:01:00.0 Off |                  N/A |
| N/A   41C    P8              1W /   80W |    5151MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----